<script src="{{site.baseurl}}/assets/js/code-runner-analytics.js"></script>

## Popcorn Hack 1: Backend Create Rule

New records are only created when the spec number is new **and** all of the required fields pass. I check the rule with an invalid record, a duplicate record, and a valid record.

In [ ]:
# CODE_RUNNER: Popcorn Hack 1 - Backend Create Rule

valid_categories = ["Auto Racing", "Drag Racing"]
existing_spec_numbers = ["1.1", "2.1"]


def can_create(record):
    fields_valid = (
        record["spec_number"].strip() != ""
        and record["product_name"].strip() != ""
        and record["effective_date"].strip() != ""
        and record["category"] in valid_categories
    )
    duplicate_spec = record["spec_number"] in existing_spec_numbers
    print("  fields_valid:", fields_valid, "| duplicate_spec:", duplicate_spec)
    # Create the record only if the fields pass AND the spec is NOT already stored.
    return fields_valid and not duplicate_spec


invalid_part = {
    "product_name": "Street Flywheel",
    "category": "Street Car",
    "spec_number": "3.1",
    "effective_date": "Jan. 1, 2026"
}

duplicate_part = {
    "product_name": "Racing Flywheel Record",
    "category": "Auto Racing",
    "spec_number": "2.1",
    "effective_date": "Mar. 3, 2010"
}

valid_part = {
    "product_name": "Multiple Disc Clutch Assemblies",
    "category": "Drag Racing",
    "spec_number": "1.2",
    "effective_date": "Feb. 9, 2006"
}

tests = [("invalid", invalid_part), ("duplicate", duplicate_part), ("valid", valid_part)]

for label, record in tests:
    print("Checking (" + label + "):", record["spec_number"], "-", record["product_name"])
    if can_create(record):
        print("  -> CREATE it")
    else:
        print("  -> REJECT it")

## Popcorn Hack 2: SFI Search Filter

A record matches if the query shows up in its product name **or** is exactly its spec number, but it is only returned if its racing category is accepted. I test a few different queries to prove the search still works when the query is swapped out.

In [ ]:
# CODE_RUNNER: Popcorn Hack 2 - SFI Search Filter

query = "racing"

records = [
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    },
    {
        "product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    }
]

valid_categories = ["Auto Racing", "Drag Racing"]


def search(search_query):
    results = []
    for record in records:
        accepted_category = record["category"] in valid_categories
        name_match = search_query.lower() in record["product_name"].lower()
        spec_match = search_query == record["spec_number"]
        # accepted category AND (name OR spec)
        if accepted_category and (name_match or spec_match):
            results.append(record)
    return results


for test_query in [query, "1.1", "assemblies", "0.0"]:
    matches = search(test_query)
    print('Query "' + test_query + '" returned', len(matches), "match(es)")
    for record in matches:
        print("  ", record["spec_number"], "-", record["product_name"])

## Popcorn Hack 3: Check One SFI Part Record

For a record to be complete, it needs a spec number, an effective date, a product name, and a racing category. Only Auto Racing and Drag Racing are allowed as categories. The runner shows whether each rule is True or False.

In [ ]:
# CODE_RUNNER: Popcorn Hack 3 - Check SFI Part Fields

part = {
    "product_name": "Racing Flywheel Record",
    "category": "Auto Racing",
    "spec_number": "2.1",
    "effective_date": "Mar. 3, 2010"
}

valid_categories = ["Auto Racing", "Drag Racing"]

# A separate Boolean for every rule.
has_spec_number = part["spec_number"].strip() != ""
has_effective_date = part["effective_date"].strip() != ""
has_product_name = part["product_name"].strip() != ""
valid_category = part["category"] in valid_categories

print("Spec number filled in:", has_spec_number)
print("Effective date filled in:", has_effective_date)
print("Product name filled in:", has_product_name)
print("Racing category allowed:", valid_category)

# The record is complete only when all four rules are True.
record_is_complete = has_spec_number and has_effective_date and has_product_name and valid_category
print("Record is complete:", record_is_complete)

## Popcorn Hack 4 / Homework: SFI Backend Validator

A validator that can be reused for any new record. To pass, a record needs a spec number that is not stored yet, an effective date, a product name, and an allowed racing category. When a record fails, the reasons are printed next to it.

In [ ]:
# CODE_RUNNER: Popcorn Hack 4 / Homework - SFI Backend Validator

existing_spec_numbers = ["1.1", "2.1"]
valid_categories = ["Auto Racing", "Drag Racing"]

test_records = [
    {
        "product_name": "Copy of Flywheel Record",
        "category": "Drag Racing",
        "spec_number": "1.1",
        "effective_date": "Jan. 1, 2026"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2",
        "effective_date": "Feb. 9, 2006"
    },
    {
        "product_name": "Replacement Flywheels",
        "category": "Street Car",
        "spec_number": "3.1",
        "effective_date": ""
    }
]


def validate_record(record):
    has_spec_number = record["spec_number"].strip() != ""
    duplicate_spec = record["spec_number"] in existing_spec_numbers
    has_effective_date = record["effective_date"].strip() != ""
    has_product_name = record["product_name"].strip() != ""
    valid_category = record["category"] in valid_categories

    is_valid = (
        has_spec_number
        and not duplicate_spec
        and has_effective_date
        and has_product_name
        and valid_category
    )

    reasons = []
    if not has_spec_number:
        reasons.append("spec number is blank")
    if duplicate_spec:
        reasons.append("spec " + record["spec_number"] + " is taken")
    if not has_effective_date:
        reasons.append("effective date is blank")
    if not has_product_name:
        reasons.append("product name is blank")
    if not valid_category:
        reasons.append("category '" + record["category"] + "' is not allowed")

    return is_valid, reasons


for record in test_records:
    is_valid, reasons = validate_record(record)
    if is_valid:
        existing_spec_numbers.append(record["spec_number"])  # store it
        print("PASS:", record["spec_number"], "-", record["product_name"])
    else:
        print("FAIL:", record["spec_number"], "-", record["product_name"], "->", ", ".join(reasons))

print()
print("Spec numbers in storage:", existing_spec_numbers)